# Customer Engagement & Retention Analytics
**Miriam Garcia** | [miriamgarcia.org](http://miriamgarcia.org) | [GitHub](https://github.com/magg6789) | [LinkedIn](https://linkedin.com/in/miriamgarcia)

---

## Business Context
Customer retention is one of the most direct levers for e-commerce revenue growth. Acquiring a new customer costs significantly more than keeping an existing one, yet most platforms lose the majority of customers after a single purchase.

This project analyzes the **Olist Brazilian E-Commerce dataset** to understand where customers are dropping off and what the data says about why.

**Questions this analysis answers:**
- What does the repeat purchase and churn profile look like?
- How do customer cohorts retain over time?
- Is churn driven by poor satisfaction, or something else?
- Where does the engagement funnel break down?
- What does revenue trajectory look like and what drives it?

**Dataset:** Olist Brazilian E-Commerce (Kaggle) — 100K+ orders, 96K customers, 2016–2018  
**Tools:** Python (Pandas, NumPy, Matplotlib, Seaborn), SQL-style transformations, Tableau

## 0. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Color palette
GREEN = '#2C5F2D'
LIME  = '#97BC62'
GOLD  = '#F5C842'
RED   = '#B85042'
GRAY  = '#6B6B6B'

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f9f9f9',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

# Load datasets
orders      = pd.read_csv('olist_orders_dataset.csv', parse_dates=[
                  'order_purchase_timestamp', 'order_approved_at',
                  'order_delivered_customer_date', 'order_estimated_delivery_date'])
customers   = pd.read_csv('olist_customers_dataset.csv')
items       = pd.read_csv('olist_order_items_dataset.csv')
reviews     = pd.read_csv('olist_order_reviews_dataset.csv')
payments    = pd.read_csv('olist_order_payments_dataset.csv')
products    = pd.read_csv('olist_products_dataset.csv')
translation = pd.read_csv('product_category_name_translation.csv')

print(f"Orders:    {orders.shape[0]:,}")
print(f"Customers: {customers['customer_unique_id'].nunique():,} unique")
print(f"Items:     {items.shape[0]:,}")
print(f"Date range: {orders['order_purchase_timestamp'].min().date()} to {orders['order_purchase_timestamp'].max().date()}")

## 1. Data Preparation

Filtering to delivered orders only, then merging customers, order value, and reviews into a single working table.

In [ ]:
# Filter to delivered orders only
orders_clean = orders[orders['order_status'] == 'delivered'].copy()
orders_clean['year_month'] = orders_clean['order_purchase_timestamp'].dt.to_period('M')

# Merge unique customer ID and state
orders_clean = orders_clean.merge(
    customers[['customer_id', 'customer_unique_id', 'customer_state']],
    on='customer_id', how='left')

# Order-level revenue and item count
order_value = items.groupby('order_id').agg(
    revenue=('price', 'sum'),
    items_count=('order_item_id', 'count')
).reset_index()
orders_clean = orders_clean.merge(order_value, on='order_id', how='left')

# Review scores (one per order)
reviews_clean = reviews[['order_id', 'review_score']].drop_duplicates('order_id')
orders_clean  = orders_clean.merge(reviews_clean, on='order_id', how='left')

# English product categories
products_en = products.merge(translation, on='product_category_name', how='left')
items_full  = items.merge(
    products_en[['product_id', 'product_category_name_english']], on='product_id', how='left')

print(f"Clean dataset: {orders_clean.shape[0]:,} delivered orders")
print(f"Date range: {orders_clean['order_purchase_timestamp'].min().date()} to {orders_clean['order_purchase_timestamp'].max().date()}")
orders_clean.head(3)

## 2. KPI Framework

Defining the core metrics before any segmentation.

In [ ]:
repeat_customers = orders_clean.groupby('customer_unique_id')['order_id'].count()
repeat_rate      = (repeat_customers > 1).sum() / orders_clean['customer_unique_id'].nunique() * 100

kpis = {
    'Total Unique Customers': orders_clean['customer_unique_id'].nunique(),
    'Total Delivered Orders':  orders_clean['order_id'].nunique(),
    'Total Revenue (BRL)':     orders_clean['revenue'].sum(),
    'Avg Order Value (BRL)':   orders_clean['revenue'].mean(),
    'Repeat Customer Rate':    repeat_rate,
    'Avg Review Score':        orders_clean['review_score'].mean(),
}

print("=" * 45)
for k, v in kpis.items():
    if 'Revenue' in k and 'Avg' not in k:
        print(f"  {k:<30} R${v:,.0f}")
    elif 'Avg Order' in k:
        print(f"  {k:<30} R${v:.2f}")
    elif 'Rate' in k:
        print(f"  {k:<30} {v:.1f}%")
    elif 'Score' in k:
        print(f"  {k:<30} {v:.2f} / 5.0")
    else:
        print(f"  {k:<30} {v:,.0f}")
print("=" * 45)

## 3. Cohort Retention Analysis

Each customer is assigned to the cohort of their first purchase month. Retention is then tracked as the share of that cohort who place at least one order in each subsequent month.

A well-retained cohort holds 20-30%+ by month 3. What we see here is a different story.

In [ ]:
# First purchase month per customer
cohort_data = orders_clean.groupby('customer_unique_id')['order_purchase_timestamp'].min().reset_index()
cohort_data.columns = ['customer_unique_id', 'first_purchase']
cohort_data['cohort'] = cohort_data['first_purchase'].dt.to_period('M')

# Merge cohort back to orders
orders_cohort = orders_clean.merge(cohort_data[['customer_unique_id', 'cohort']], on='customer_unique_id')
orders_cohort['order_period']  = orders_cohort['order_purchase_timestamp'].dt.to_period('M')
orders_cohort['period_number'] = (orders_cohort['order_period'] - orders_cohort['cohort']).apply(lambda x: x.n)

# Build cohort retention table
cohort_table = orders_cohort.groupby(['cohort', 'period_number'])['customer_unique_id'].nunique().reset_index()
cohort_pivot = cohort_table.pivot(index='cohort', columns='period_number', values='customer_unique_id')
cohort_sizes = cohort_pivot[0]
cohort_pct   = cohort_pivot.divide(cohort_sizes, axis=0) * 100

# Plot 2017 cohorts, first 13 periods
cohort_plot = cohort_pct[cohort_pct.index.year == 2017].iloc[:, :13].copy()
cohort_plot.index = cohort_plot.index.astype(str)

fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(cohort_plot, annot=True, fmt='.1f', mask=cohort_plot.isna(),
            cmap='YlGn', linewidths=0.5, linecolor='white', ax=ax,
            cbar_kws={'label': 'Retention %'}, annot_kws={'size': 8})
ax.set_title('Cohort Retention Analysis — 2017 Customer Cohorts\nRetention % by Months Since First Purchase',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Months Since First Purchase')
ax.set_ylabel('Acquisition Cohort')
ax.tick_params(axis='x', rotation=0)
ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.show()

print("\nRetention drops to ~0.5% by month 1 across all 2017 cohorts.")
print("Almost no customers return for a second purchase within the observed window.")

## 4. Churn Analysis

Defining churn as 180 days of inactivity after a customer's last purchase. Then checking whether churn correlates with satisfaction.

In [ ]:
# Churn classification
last_purchase = orders_clean.groupby('customer_unique_id')['order_purchase_timestamp'].max().reset_index()
last_purchase.columns = ['customer_unique_id', 'last_purchase']
dataset_end = orders_clean['order_purchase_timestamp'].max()
last_purchase['days_since_last'] = (dataset_end - last_purchase['last_purchase']).dt.days
last_purchase['churned'] = last_purchase['days_since_last'] > 180

churned = last_purchase['churned'].sum()
active  = (~last_purchase['churned']).sum()
print(f"Churned: {churned:,} ({churned/len(last_purchase)*100:.1f}%)")
print(f"Active:  {active:,} ({active/len(last_purchase)*100:.1f}%)")

# Churn by review score
churn_review   = orders_clean.merge(last_purchase[['customer_unique_id', 'churned']], on='customer_unique_id')
churn_by_score = churn_review.groupby('review_score')['churned'].mean() * 100
print("\nChurn rate by review score:")
for score, rate in churn_by_score.items():
    print(f"  Score {int(score)}: {rate:.1f}%")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sizes  = [active, churned]
labels = [f'Active\n{active:,}\n(41.1%)', f'Churned\n{churned:,}\n(58.9%)']
axes[0].pie(sizes, labels=labels, colors=[GREEN, RED], startangle=90,
            wedgeprops=dict(width=0.5), textprops={'fontsize': 11})
axes[0].set_title('Customer Churn Status\n(180-day inactivity threshold)', fontweight='bold')

axes[1].hist(last_purchase['days_since_last'], bins=40, color=GREEN, alpha=0.8, edgecolor='white')
axes[1].axvline(180, color=RED, linestyle='--', linewidth=2, label='Churn threshold (180 days)')
axes[1].set_xlabel('Days Since Last Purchase')
axes[1].set_ylabel('Number of Customers')
axes[1].set_title('Distribution of Days Since Last Purchase', fontweight='bold')
axes[1].legend(fontsize=9)

fig.suptitle('Churn Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nScore 5 customers churn at 57.5%, Score 1 customers at 60.2%.")
print("Satisfaction and retention are nearly uncorrelated. The problem is not experience quality.")

## 5. Engagement Funnel Analysis

In [ ]:
funnel = {
    'All Orders':       orders['order_id'].nunique(),
    'Delivered':        orders_clean['order_id'].nunique(),
    'With Review':      orders_clean[orders_clean['review_score'].notna()]['order_id'].nunique(),
    'Score 4 or 5':     orders_clean[orders_clean['review_score'] >= 4]['order_id'].nunique(),
    'Repeat Customers': int((repeat_customers > 1).sum()),
}

print(f"{'Stage':<25} {'Count':>8}   {'Drop':>10}")
print("-" * 48)
prev = None
for stage, count in funnel.items():
    if prev:
        drop = f"{(1 - count/prev)*100:.1f}% drop"
        print(f"{stage:<25} {count:>8,}   {drop:>10}")
    else:
        print(f"{stage:<25} {count:>8,}")
    prev = count

# Top categories
top_cats = items_full.groupby('product_category_name_english')['order_id'].nunique().nlargest(10).reset_index()
top_cats.columns = ['category', 'orders']
top_cats_sorted = top_cats.sort_values('orders')

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
funnel_colors = [GREEN, LIME, GOLD, '#E8A838', RED]
bars = axes[0].barh(range(len(funnel)), list(funnel.values()),
                    color=funnel_colors, alpha=0.85, height=0.6)
axes[0].set_yticks(range(len(funnel)))
axes[0].set_yticklabels(list(funnel.keys()), fontsize=9)
axes[0].set_xlabel('Number of Orders / Customers')
axes[0].set_title('Engagement Funnel', fontweight='bold')
axes[0].xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'{v:,.0f}'))
axes[0].invert_yaxis()
for bar, val in zip(bars, funnel.values()):
    axes[0].text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2,
                 f'{val:,}', va='center', fontsize=8)

axes[1].barh(top_cats_sorted['category'], top_cats_sorted['orders'], color=GREEN, alpha=0.8)
axes[1].set_xlabel('Number of Orders')
axes[1].set_title('Top 10 Product Categories by Orders', fontweight='bold')
axes[1].xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'{v:,.0f}'))

fig.suptitle('Engagement Funnel & Category Breakdown', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nThe biggest drop is from satisfied customers to repeat purchasers: 96% falloff.")
print("Acquisition is working. Getting customers to come back is the core problem to solve.")

## 6. Revenue Trend & Month-over-Month Growth

In [ ]:
monthly = orders_clean.groupby('year_month').agg(
    revenue=('revenue', 'sum'),
    orders=('order_id', 'nunique'),
    customers=('customer_unique_id', 'nunique')
).reset_index()
monthly = monthly[monthly['year_month'].astype(str) >= '2017-01'].copy()
monthly['year_month_str'] = monthly['year_month'].astype(str)
monthly['mom_growth'] = monthly['revenue'].pct_change() * 100

fig, ax1 = plt.subplots(figsize=(14, 5))
x = range(len(monthly))

bar_colors = [LIME if g >= 0 else RED for g in monthly['mom_growth'].fillna(0)]
ax1.bar(x, monthly['revenue'] / 1000, color=bar_colors, alpha=0.85, width=0.6)
ax1.set_ylabel('Revenue (BRL thousands)')
ax1.set_xticks(x)
ax1.set_xticklabels(monthly['year_month_str'], rotation=45, ha='right', fontsize=8)
ax1.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'R${v:.0f}K'))

ax2 = ax1.twinx()
ax2.plot(x, monthly['mom_growth'], color=GREEN, linewidth=2, marker='o', markersize=5)
ax2.axhline(0, color=GRAY, linestyle='--', linewidth=0.8)
ax2.set_ylabel('Month-over-Month Growth %')
ax2.yaxis.set_major_formatter(mtick.PercentFormatter())
ax2.spines['top'].set_visible(False)

nov_idx = list(monthly['year_month_str']).index('2017-11')
ax1.annotate('Black Friday +52% MoM',
             xy=(nov_idx, monthly.iloc[nov_idx]['revenue'] / 1000),
             xytext=(nov_idx + 1.5, monthly.iloc[nov_idx]['revenue'] / 1000 + 60),
             arrowprops=dict(arrowstyle='->', color=RED),
             fontsize=8, color=RED)

green_patch = mpatches.Patch(color=LIME, label='Revenue (growth month)')
red_patch   = mpatches.Patch(color=RED,  label='Revenue (decline month)')
ax1.legend(handles=[green_patch, red_patch], loc='upper left', fontsize=9)
ax1.set_title('Monthly Revenue & MoM Growth — Jan 2017 to Aug 2018', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Key Findings & Recommendations

### Findings

| Metric | Value | Interpretation |
|--------|-------|----------------|
| Repeat Customer Rate | 3.0% | Most customers purchase once. Acquisition is working, retention is not. |
| Churn Rate (180d) | 58.9% | More than half of customers go inactive within 6 months of their first purchase. |
| Avg Review Score | 4.16 / 5.0 | Customers are satisfied. Churn is not a quality problem. |
| Score 5 vs Score 1 Churn | 57.5% vs 60.2% | Satisfied customers churn only marginally less than dissatisfied ones. |
| Black Friday Spike | +52% MoM Nov 2017 | Revenue is heavily seasonal with a clear dependency on promotional events. |
| Cohort Retention (Month 1) | ~0.5% | Almost no customers come back for a second purchase within their first month. |

### Recommendations

1. **Post-purchase re-engagement:** The gap between satisfied customers (83% score 4 or 5) and repeat customers (3%) is where revenue is being lost. A targeted re-engagement sequence at 30, 60, and 90 days post-purchase is the most direct lever to pull.

2. **Satisfaction alone does not drive loyalty:** A 4.16 average review score with 58.9% churn tells a clear story. The experience is good but there is no mechanism pulling customers back. Loyalty programs, personalized recommendations, or subscription options would directly address this.

3. **Seasonal revenue concentration:** November 2017 accounted for a disproportionate share of annual revenue. Off-peak campaigns in Q1 and Q3 would reduce that dependency and smooth out the revenue curve.

4. **Category-level cohort analysis:** Bed/bath, sports, and electronics likely have different natural repurchase cycles. A category-level cohort cut would help prioritize which segments to target first for retention efforts.

### Next Steps
- Build a churn prediction model using customer features: tenure, AOV, review score, category mix
- A/B test re-engagement email timing at 30, 60, and 90 days post-purchase
- Segmentation analysis to identify high-LTV customer profiles for acquisition targeting

---
*Dataset: [Olist Brazilian E-Commerce](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) via Kaggle*

If you made it this far send me a "hi" on Linkedin! 